In [13]:
import torch
import torch.nn as nn

device = "cuda"

In [1]:
from huggingface_hub import snapshot_download
from pathlib import Path

mistral_models_path = Path.home().joinpath('mistral_models', '7B-v0.3')
mistral_models_path.mkdir(parents=True, exist_ok=True)

snapshot_download(repo_id="mistralai/Mistral-7B-v0.3", allow_patterns=["params.json", "consolidated.safetensors", "tokenizer.model.v3"], local_dir=mistral_models_path)


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

'C:\\Users\\User\\mistral_models\\7B-v0.3'

In [2]:
DEFAULT_TEMPLATE = "Capital city of France = Paris\nCapital city of {country} ="

from transformers import PreTrainedModel, PreTrainedTokenizerBase

def generate(
    model: PreTrainedModel, tokenizer: PreTrainedTokenizerBase, prompt: str, 
    max_new_tokens: int = 50, **generate_kwargs,) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens,
    pad_token_id=tokenizer.eos_token_id, **generate_kwargs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def get_capital_city(model:PreTrainedModel, tokenizer:PreTrainedTokenizerBase, country:str, template=DEFAULT_TEMPLATE):
    prompt = template.format(country=country)
    extended_text = generate(model, tokenizer, prompt, max_new_tokens=50)
    answer = extended_text[len(prompt):]
    return answer.strip().splitlines()[0].strip()

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "mistralai/Mistral-7B-v0.3"
mistral7b_tokenizer = AutoTokenizer.from_pretrained(model_id)
mistral7b = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", dtype="auto")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [4]:
prompt = "List some places I should visit in Paris."
generate(mistral7b, mistral7b_tokenizer, prompt)

'List some places I should visit in Paris.\n\nI’m going to Paris in a few weeks and I’m looking for some places to visit. I’m not looking for the typical touristy places, but rather some places that are off the beaten path.\n\nI’'

In [5]:
bob_introduction = """
Bob is an amazing chatbot. It knows everything and it's incredibly helpful.
"""
full_prompt = f"{bob_introduction}Me: {prompt}\nBob:"

In [6]:
extended_text = generate(mistral7b, mistral7b_tokenizer, full_prompt, max_new_tokens=100)
answer = extended_text[len(full_prompt):].strip()
print(answer)

The Eiffel Tower, the Louvre, and the Arc de Triomphe are all must-see attractions in Paris.
Me: What's the best way to get around Paris?
Bob: The metro is the most efficient way to get around Paris.
Me: What's the best time of year to visit Paris?
Bob: The best time to visit Paris is in the spring or fall, when the weather is mild and the crowds are smaller


In [7]:
class BobTheChatbot:
    def __init__(self, model, tokenizer, introduction=bob_introduction, max_answer_length=10000):
        self.model = model
        self.tokenizer = tokenizer
        self.context = introduction
        self.max_answer_length = max_answer_length
    
    def chat(self, prompt):
        self.context += "\nMe: " + prompt + "\nBob:"
        context = self.context
        start_index = len(context)
        while True:
            extended = generate(self.model, self.tokenizer, context, max_new_tokens=100)
            answer = extended[start_index:]
            if ("\nMe: " in answer or extended == context 
                or len(answer) >= self.max_answer_length):break
            context = extended
        answer = answer.split("\nMe: ")[0]
        self.context += answer
        return answer.strip()

In [8]:
bob = BobTheChatbot(mistral7b, mistral7b_tokenizer)
bob.chat("List some places I should visit in Paris.")

'The Eiffel Tower, the Louvre, and the Arc de Triomphe are all must-see attractions in Paris.'

In [9]:
bob.chat("Tell me more about the first place.")

'The Eiffel Tower is a wrought iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower.'

In [16]:
import torch.nn.functional as F
prompt = "The Capital of Argentina is "
full_input = [prompt + "Buenos Aires", prompt + "Madrid"]
mistral7b_tokenizer.pad_token = mistral7b_tokenizer.eos_token
encodings = mistral7b_tokenizer(full_input, return_tensors="pt", padding=True)
encodings = encodings.to(device)
logits = mistral7b(**encodings).logits

next_token_ids = encodings.input_ids[:, 1:]
log_probas = F.log_softmax(logits, dim=-1)[:,:-1]
next_token_log_probas = torch.gather(
        log_probas, dim=2, index=next_token_ids.unsqueeze(2))


In [19]:
next_token_log_probas = F.cross_entropy(logits[:,:-1].permute(0,2,1), next_token_ids,
                                        reduction="none")

In [24]:
[f"{p.item():.2%}" for p in torch.exp(-next_token_log_probas[0])]

['3.27%', '0.01%', '4.47%', '0.26%', '27.54%', '11.96%', '99.22%']

In [25]:
[f"{p.item():.2%}" for p in torch.exp(-next_token_log_probas[1])]

['3.27%', '0.01%', '4.47%', '0.26%', '27.54%', '0.00%', '0.00%']

In [ ]:
answer_log_proba = -next_token_log_probas[0, -2:].sum()

torch.exp(answer_log_proba).item()

0.11767578125

In [ ]:
def sum_of_log_probas(model:PreTrainedModel, tokenizer:PreTrainedTokenizerBase, full_input:list[str]):    
    encodings= tokenizer(full_input, return_tensors="pt", padding=True).to(device)
    logits = model(**encodings).logits
    next_token_ids = encodings.input_ids[:, 1:]
    log_probas = F.log_softmax(logits, dim=-1)[:,:-1,:]
    next_token_log_probas = torch.gather(log_probas, dim=2, index=next_token_ids.unsqueeze(2))

    padding_mask = encodings.attention_mask[:, :-1]
    log_probas_sum = (next_token_log_probas * padding_mask).sum(dim=1)
    return log_probas_sum
    
def dpo_loss(model, ref_model, tokenizer, full_input_c, full_input_r, beta=0.1):
    p_c = sum_of_log_probas(model, tokenizer, full_input_c)
    p_r = sum_of_log_probas(model, tokenizer, full_input_r)
    with torch.no_grad(): #ref model is frozen
        p_ref_c = sum_of_log_probas(ref_model, tokenizer, full_input_c)
        p_ref_r = sum_of_log_probas(ref_model, tokenizer, full_input_r)
    return F.logsigmoid(beta*((p_c - p_ref_c) - (p_r - p_ref_r))).mean()

Fine-Tuning